<a href="https://colab.research.google.com/github/volodymyr-holovan/DTA2026/blob/main/homework/ml_practice_LR%26Cl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практика: лінійна регресія та класифікація

Це тренувальний блокнот для закріплення базового циклу ML. Завдання **нескладні** й повторюють кроки з основного тьюторіалу — тільки тепер усе робиш **сам**.

**Дві задачі на двох нових наборах даних:**
- **Задача A (регресія):** передбачити **зарплату** працівника.
- **Задача B (класифікація):** передбачити, чи **складе студент іспит** (так/ні).

**Як працювати:**
1. Запусти комірку «Підготовка даних» нижче — вона все налаштує.
2. Іди по кроках. Там, де стоїть `# TODO`, — впиши свій код.
3. Підказки є під кожним кроком.

> 💡 Усі потрібні інструменти ти вже бачив: `train_test_split`, `LinearRegression`, `DecisionTreeClassifier`, `.fit()`, `.predict()`, метрики. Тримай той блокнот поруч як шпаргалку.

---

## 🔧 Підготовка даних (просто запусти)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 20)

# ---------- Дані A: ЗАРПЛАТИ (для регресії) ----------
N = 800
experience = np.random.randint(0, 31, N)                      # стаж, років
age        = (22 + experience + np.random.randint(0, 12, N)).clip(22, 64)  # вік
education  = np.random.choice([12, 15, 16, 18, 20], N,        # років освіти
                              p=[.2, .15, .35, .2, .1])
english    = np.random.randint(1, 6, N)                       # рівень англ. 1..5

salary = (8000                       # базова ставка, грн
          + experience * 900         # за кожен рік стажу
          + education  * 600         # за рік освіти
          + english    * 1500        # за рівень англійської
          + np.random.normal(0, 3000, N)   # шум: усе інше
         ).clip(8000, None)

salary_df = pd.DataFrame({
    "experience": experience, "age": age,
    "education": education, "english": english,
    "salary": salary.round(0).astype(int),
})

# ---------- Дані B: СТУДЕНТИ (для класифікації) ----------
M = 800
study     = np.random.normal(12, 5, M).clip(0, 30)            # годин навчання/тиждень
attendance= np.random.normal(78, 15, M).clip(30, 100)        # відвідуваність, %
prev_score= np.random.normal(65, 18, M).clip(0, 100)         # бал за минулий іспит
sleep     = np.random.normal(7, 1.2, M).clip(4, 10)          # годин сну

score_logit = (0.12*study + 0.04*attendance + 0.05*prev_score
               + 0.3*sleep - 9 + np.random.normal(0, 1.2, M))
passed = (score_logit > 0).astype(int)                        # 1 = склав, 0 = ні

students_df = pd.DataFrame({
    "study": study.round(1), "attendance": attendance.round(0).astype(int),
    "prev_score": prev_score.round(0).astype(int), "sleep": sleep.round(1),
    "passed": passed,
})

print("✅ Дані готові.")
print("Зарплати:", salary_df.shape, "| Студенти:", students_df.shape)
print("Частка тих, хто склав іспит:", f"{students_df['passed'].mean():.0%}")

✅ Дані готові.
Зарплати: (800, 5) | Студенти: (800, 5)
Частка тих, хто склав іспит: 69%


---
# 🟦 Задача A. Регресія: передбачаємо зарплату

Дані у таблиці `salary_df`. Ознаки: `experience` (стаж), `age` (вік), `education` (років освіти), `english` (рівень англійської 1–5). Ціль: `salary` (зарплата, грн).

Мета — навчити модель передбачати зарплату і **пояснити**, що на неї впливає.

### Крок A1. Подивись на дані
Виведи перші рядки таблиці й описову статистику. Це звичка №1 перед будь-яким навчанням.

*Підказка:* `salary_df.head()` і `salary_df.describe()`.

In [3]:
# TODO: виведи перші рядки salary_df
print(salary_df.head())
# TODO: виведи describe()
print(salary_df.describe())

   experience  age  education  english  salary
0           6   32         15        2   23714
1          19   44         12        3   32814
2          28   54         12        5   54472
3          14   42         20        5   35964
4          10   40         18        2   29723
       experience         age   education     english        salary
count  800.000000  800.000000  800.000000  800.000000    800.000000
mean    15.418750   42.746250   15.856250    3.028750  36028.926250
std      9.328568    9.924338    2.408908    1.432826   9242.043432
min      0.000000   22.000000   12.000000    1.000000  12917.000000
25%      7.000000   35.000000   15.000000    2.000000  28550.750000
50%     16.000000   43.000000   16.000000    3.000000  36102.500000
75%     24.000000   51.000000   18.000000    4.000000  43545.500000
max     30.000000   62.000000   20.000000    5.000000  55947.000000


### Крок A2. Признач ознаки (X) і ціль (y), поділи на train / test
- `X` — усі стовпці, КРІМ `salary`.
- `y` — стовпець `salary`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`.

*Підказка:* `X = salary_df[["experience", "age", "education", "english"]]`,
`y = salary_df["salary"]`, далі `train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)`.

In [4]:
from sklearn.model_selection import train_test_split

# TODO: створи X та y
X = salary_df[["experience", "age", "education", "english"]]
y = salary_df["salary"]

# TODO: поділи на X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 640 | Test: 160


### Крок A3. Навчи лінійну регресію
Згадай цикл: **створити → `.fit(X_train, y_train)`**.

*Підказка:* `from sklearn.linear_model import LinearRegression`, далі `model = LinearRegression()` і `model.fit(...)`.

In [5]:
from sklearn.linear_model import LinearRegression

# TODO: створи та навчи модель
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

### Крок A4. Зроби передбачення й оціни якість
- Передбач на `X_test`.
- Порахуй **MAE** та **R²**.

*Підказка:* `y_pred = model.predict(X_test)`; `mean_absolute_error(y_test, y_pred)`;
`r2_score(y_test, y_pred)`.

In [6]:
from sklearn.metrics import mean_absolute_error, r2_score

# TODO: передбач y_pred
y_pred = model.predict(X_test)

# TODO: порахуй і виведи MAE та R²
print(mean_absolute_error(y_test, y_pred))
print(r2_score(y_test, y_pred))

2545.3311797099914
0.881453119757412


### Крок A5. 🔑 Інтерпретуй коефіцієнти
Дістань коефіцієнти моделі й скажи словами, яка ознака найсильніше підвищує зарплату.

*Підказка:* `model.coef_` і `model.intercept_`. Зістав назви з `X.columns`.

In [7]:
# TODO: побудуй таблицю "ознака — коефіцієнт" і відсортуй
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_
}).sort_values("coefficient", ascending=False)

print(coef_df)
print("Intercept:", model.intercept_)

      feature  coefficient
3     english  1597.454634
0  experience   872.476660
2   education   609.589202
1         age    31.151182
Intercept: 6850.026177239346


✍️ **Запиши відповідь словами** (просто текстом у цій комірці, подвійний клік):
> Найсильніше на зарплату впливає ознака english, бо вона має найбільший додатний коефіцієнт (1597).

### Крок A6. Передбач зарплату для нового працівника
Створи одного працівника й передбач його зарплату: стаж 5, вік 30, освіта 16, англійська 4.

*Підказка:* зроби `pd.DataFrame([{...}])` з тими самими назвами стовпців і передай у `model.predict(...)`.

In [8]:
# TODO: створи new_employee і передбач зарплату
new_employee = pd.DataFrame([{
    "experience": 5,
    "age": 30,
    "education": 16,
    "english": 4
}])

pred_salary = model.predict(new_employee)

print("Прогнозована зарплата:", pred_salary[0])

Прогнозована зарплата: 28290.1906983186


---
# 🟩 Задача B. Класифікація: чи складе студент іспит

Дані у таблиці `students_df`. Ознаки: `study` (годин навчання/тиждень), `attendance` (відвідуваність %), `prev_score` (бал за минулий іспит), `sleep` (годин сну). Ціль: `passed` (1 = склав, 0 = ні).

### Крок B1. Подивись на дані
Виведи перші рядки й перевір баланс класів: яка частка студентів склала іспит?

*Підказка:* `students_df.head()` і `students_df["passed"].mean()`.

In [9]:
# TODO: head() і частка тих, хто склав
print(students_df.head())

pass_rate = students_df["passed"].mean()
print("Частка тих, хто склав:", pass_rate)

   study  attendance  prev_score  sleep  passed
0   11.2          76          94    5.7       1
1   18.5          72          49    9.4       1
2   15.7          71          59    7.9       0
3   16.1         100          88    5.1       1
4    9.9          92          72    7.0       1
Частка тих, хто склав: 0.69375


### Крок B2. X, y і поділ на train / test
- `X` — усе, крім `passed`. `y` — `passed`.
- Додай `stratify=y`, щоб пропорція класів збереглася.

*Підказка:* `train_test_split(Xs, ys, test_size=0.2, random_state=RANDOM_STATE, stratify=ys)`.

In [10]:
# TODO: Xs, ys та поділ на Xs_train, Xs_test, ys_train, ys_test
Xs = students_df.drop(columns=["passed"])
ys = students_df["passed"]

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    Xs,
    ys,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=ys
)

print("Train:", Xs_train.shape[0])
print("Test:", Xs_test.shape[0])

Train: 640
Test: 160


### Крок B3. Навчи дерево рішень
Використай `DecisionTreeClassifier` з `max_depth=3` (щоб було просте й читабельне) і `random_state=RANDOM_STATE`.

*Підказка:* `from sklearn.tree import DecisionTreeClassifier`.

In [11]:
from sklearn.tree import DecisionTreeClassifier

# TODO: створи та навчи дерево
tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE
)

tree.fit(Xs_train, ys_train)

DecisionTreeClassifier(max_depth=3, random_state=42)

### Крок B4. Передбач і оціни
- Передбач на `Xs_test`.
- Порахуй **accuracy** і побудуй **матрицю плутанини**.

*Підказка:* `accuracy_score(ys_test, ys_pred)` та `confusion_matrix(ys_test, ys_pred)`.

In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix

# TODO: передбач ys_pred, порахуй accuracy та матрицю плутанини
ys_pred = tree.predict(Xs_test)

acc = accuracy_score(ys_test, ys_pred)
cm = confusion_matrix(ys_test, ys_pred)

print("Accuracy:", acc)
print("Confusion matrix:")
print(cm)

Accuracy: 0.75625
Confusion matrix:
[[25 24]
 [15 96]]


### Крок B5. Що найбільше впливає на результат?
Виведи важливість ознак дерева й назви найважливішу.

*Підказка:* `tree.feature_importances_`, зістав із `Xs.columns`.

In [13]:
# TODO: таблиця "ознака — важливість", відсортована за спаданням
importance_df = pd.DataFrame({
    "feature": Xs.columns,
    "importance": tree.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df)

      feature  importance
2  prev_score    0.377026
0       study    0.325006
1  attendance    0.195933
3       sleep    0.102035


✍️ **Відповідь словами:**
> Найбільше на складання іспиту впливає prev_score.

### Крок B6. Передбач для нового студента
Студент: навчання 15 год, відвідуваність 85%, минулий бал 70, сон 7.5.
Виведи і рішення (`predict`), і **ймовірність** скласти (`predict_proba`).

*Підказка:* `predict_proba(...)[0, 1]` — це ймовірність класу «склав».

In [14]:
# TODO: створи new_student, виведи рішення та ймовірність
new_student = pd.DataFrame([{
    "study": 15,
    "attendance": 85,
    "prev_score": 70,
    "sleep": 7.5
}])

prediction = tree.predict(new_student)[0]
probability = tree.predict_proba(new_student)[0, 1]

print("Складе іспит:", prediction)
print("Ймовірність скласти:", probability)

Складе іспит: 1
Ймовірність скласти: 0.8805460750853242


---
# ⭐ Бонус (необов'язково, але корисно)

1. **Перевір на перенавчання.** Для дерева зі Задачі B порахуй accuracy окремо на `Xs_train` і на `Xs_test`. Великий розрив = зубріння. Потім спробуй `max_depth=10` — розрив зросте?
2. **Сильніша модель.** Навчи `RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)` на тих самих даних і порівняй accuracy з деревом.
3. **Прибери ознаку.** У Задачі A прибери `experience` з `X`, перенавчи й подивись, як впаде R². Який висновок про важливість стажу?

In [15]:
# Місце для бонусних експериментів

---
# 🧠 Питання на розуміння (без коду)

Дай собі відповідь словами (правильні — у блокноті з розв'язками):
1. Чому ми оцінюємо модель на `X_test`, а не на `X_train`?
2. Задача «передбачити кількість проданих квитків» — це регресія чи класифікація? А «спам / не спам»?
3. Що означає R² = 0.85 простими словами?
4. Чому accuracy може бути оманливою, якщо лише 5% студентів провалюють іспит?
5. Коефіцієнт `english = +1500`. Як прочитати це вголос для керівника?

Відповіді
1. `X_train` це навчальна вибірка для якої модель вже знає правильні відповіді. Немає сенсу давати її ті самі дані по колу  
2. «передбачити кількість проданих квитків» => Регресія  
«спам / не спам» => Класифікація
3. 85% різниці між зарплатами людей пояснюється ознаками, які ми дали моделі. Решта 15% — вплив інших факторів
4. Якщо лише 5% студентів провалюють, то модель, яка завжди відповідає "склав", матиме accuracy 95% — і при цьому жодного разу не виявить провалу. У таких випадках потрібні precision, recall або F1-score.
5. Кожен додатковий рівень англійської мови пов'язаний із зарплатою вищою на 1 500 грн — за однакового стажу, освіти та віку.

> 🎯 Якщо впорався із задачами A і B без підглядання — ти впевнено володієш базовим циклом ML. Вітаю!